In [41]:
import torch
from torch import nn
from transformers import RobertaTokenizer, RobertaForSequenceClassification
import matplotlib.pyplot as plt

# Step 1: Define the CodeBERT model with Dropout
class CodeBERTClassifier(nn.Module):
    def __init__(self):
        super(CodeBERTClassifier, self).__init__()
        self.model = RobertaForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)
        self.dropout = nn.Dropout(p=0.3)  # Add a dropout layer

    def forward(self, input_ids, attention_mask=None):
        outputs = self.model(input_ids, attention_mask=attention_mask)
        logits = self.dropout(outputs.logits)  # Apply dropout
        return logits

# Step 2: Load the trained model
model = CodeBERTClassifier()
model.load_state_dict(torch.load('E:/python-projects/llm/Trained_models/codebert_model.pth'))
model.eval()  # Set the model to evaluation mode

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

# Step 3: Load the tokenizer
tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

# Step 4: Preprocess the input code samples
def preprocess_input_code(code_samples):
    tokenized_samples = []
    attention_masks = []

    for code_sample in code_samples:
        tokenized_input = tokenizer(
            code_sample,
            padding='max_length',
            truncation=True,
            max_length=512,
            return_tensors='pt'
        )
        tokenized_samples.append(tokenized_input['input_ids'].squeeze(0))
        attention_masks.append(tokenized_input['attention_mask'].squeeze(0))

    # Convert to PyTorch tensors
    tokens = torch.stack(tokenized_samples)
    masks = torch.stack(attention_masks)

    return tokens, masks

# Step 5: Make predictions
def predict_code_samples(model, code_samples):
    tokens, masks = preprocess_input_code(code_samples)
    
    # Move input tensors to the same device as the model
    tokens = tokens.to(device)
    masks = masks.to(device)

    with torch.no_grad():
        outputs = model(tokens, attention_mask=masks)
        probabilities = torch.nn.functional.softmax(outputs, dim=1)

    return probabilities.cpu().numpy()
    #     _, preds = torch.max(outputs, dim=1)

    # return preds.cpu().numpy()  # Return predictions as numpy array

# Load the text file with multiple code snippets
with open("sample_code.txt", "r") as file:
    file_content = file.read()

# Split based on delimiter '###'
code_samples = file_content.split("####")
code_samples = [code.strip() for code in code_samples if code.strip()]  # Clean up

# # Make predictions
# predictions = predict_code_samples(model,code_samples)
# print(predictions)

# # Print results
# # for code, prediction in zip(new_code_samples, predictions):
# for idx, (code, prediction) in enumerate(zip(code_samples, predictions)):
#     label = "AI-generated" if prediction == 1 else "Human-generated"
#     print(f"Sample {idx+1}:")
#     print(f"Prediction: {label}\n")

# Make predictions
probabilities = predict_code_samples(model, code_samples)

# Print results with percentages for each code sample
for idx, (code, prob) in enumerate(zip(code_samples, probabilities)):
    ai_generated_prob = prob[1] * 100  # Percentage for AI-generated class (label 1)
    human_generated_prob = prob[0] * 100  # Percentage for Human-written class (label 0)
    print(f"Sample {idx+1}:")
    print(f"Prediction: {ai_generated_prob:.2f}% AI-generated, {human_generated_prob:.2f}% Human-generated\n")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 124647170
Sample 1:
Prediction: 1.11% AI-generated, 98.89% Human-generated

Sample 2:
Prediction: 0.14% AI-generated, 99.86% Human-generated

Sample 3:
Prediction: 99.52% AI-generated, 0.48% Human-generated

Sample 4:
Prediction: 99.99% AI-generated, 0.01% Human-generated

Sample 5:
Prediction: 0.15% AI-generated, 99.85% Human-generated

Sample 6:
Prediction: 0.01% AI-generated, 99.99% Human-generated

Sample 7:
Prediction: 0.02% AI-generated, 99.98% Human-generated

Sample 8:
Prediction: 82.41% AI-generated, 17.59% Human-generated

Sample 9:
Prediction: 37.42% AI-generated, 62.58% Human-generated

Sample 10:
Prediction: 99.80% AI-generated, 0.20% Human-generated



In [ ]:
codebert_classifier0.8872217439429344
Total parameters: 124647170
Sample 1:
Prediction: 99.72% AI-generated, 0.28% Human-generated

Sample 2:
Prediction: 99.86% AI-generated, 0.14% Human-generated

Sample 3:
Prediction: 99.96% AI-generated, 0.04% Human-generated

Sample 4:
Prediction: 99.97% AI-generated, 0.03% Human-generated

Sample 5:
Prediction: 85.95% AI-generated, 14.05% Human-generated

In [ ]:
codebert_classifier0.8958109559613319
Total parameters: 124647170
Sample 1:
Prediction: 0.80% AI-generated, 99.20% Human-generated

Sample 2:
Prediction: 4.32% AI-generated, 95.68% Human-generated

Sample 3:
Prediction: 99.23% AI-generated, 0.77% Human-generated

Sample 4:
Prediction: 99.75% AI-generated, 0.25% Human-generated

Sample 5:
Prediction: 0.62% AI-generated, 99.38% Human-generated

In [ ]:
codebert_classifier0.9280343716433942
Total parameters: 124647170
Sample 1:
Prediction: 0.59% AI-generated, 99.41% Human-generated

Sample 2:
Prediction: 53.44% AI-generated, 46.56% Human-generated

Sample 3:
Prediction: 99.35% AI-generated, 0.65% Human-generated

Sample 4:
Prediction: 99.67% AI-generated, 0.33% Human-generated

Sample 5:
Prediction: 0.65% AI-generated, 99.35% Human-generated

In [ ]:
codebert_model.pth
Total parameters: 124647170
Sample 1:
Prediction: 1.11% AI-generated, 98.89% Human-generated

Sample 2:
Prediction: 0.14% AI-generated, 99.86% Human-generated

Sample 3:
Prediction: 99.52% AI-generated, 0.48% Human-generated

Sample 4:
Prediction: 99.99% AI-generated, 0.01% Human-generated

Sample 5:
Prediction: 0.15% AI-generated, 99.85% Human-generated